In [ ]:
# Install required libraries
import sys

!{sys.executable} -m pip install pypdf
!{sys.executable} -m pip install -U sentence-transformers
!{sys.executable} -m pip install faiss-cpu
!{sys.executable} -m pip install -U google-genai

In [2]:
# Import libraries and configure Gemini API
import os
import numpy as np
import faiss
import re

from getpass import getpass
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("Setup completed successfully.")

Enter your Gemini API key:  ········


Setup completed successfully.


In [3]:
# Load PDF documents
documents = []

for filename in os.listdir("."):
    
    if filename.lower().endswith(".pdf"):
        
        pdf_path = os.path.join(".", filename)
        reader = PdfReader(pdf_path)
        
        text = ""
        
        for page in reader.pages:
            page_text = page.extract_text()
            
            if page_text:
                text += page_text + "\n"
        
        documents.append({
            "filename": filename,
            "text": text
        })

print(f"Loaded {len(documents)} PDF files.")

for doc in documents:
    print(doc["filename"], "→", len(doc["text"]), "characters")

Loaded 7 PDF files.
backyard-beekeeping.pdf → 11507 characters
HBHC-Guide_Varroa-Management.pdf → 80725 characters
HBHC_Hive_BMPs_for_ Hive_Health.pdf → 179217 characters
Honey Bee Maintenance Expenses.pdf → 15353 characters
Residential Beekeeping.pdf → 58301 characters
The Basics of Queen Management.pdf → 14734 characters
Thriving Hives_ The Apiary - Beekeeping Essentials.pdf → 24513 characters


In [4]:
# Clean extracted text
for doc in documents:
    text = doc["text"]
    
    # Replace multiple spaces/tabs with a single space
    text = re.sub(r"[ \t]+", " ", text)
    
    # Remove excessive blank lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    
    # Remove spaces at the beginning/end of lines
    text = "\n".join(line.strip() for line in text.splitlines())
    
    doc["clean_text"] = text

print("Cleaning completed.")

Cleaning completed.


In [5]:
# Split documents into overlapping chunks
def create_chunks(text, chunk_size=500, overlap=50):
    chunks = []
    
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        
        chunk = text[start:end]
        chunks.append(chunk)
        
        start += chunk_size - overlap
    
    return chunks

all_chunks = []

for doc in documents:
    
    if doc["clean_text"].strip():
        
        chunks = create_chunks(doc["clean_text"])
        
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "text": chunk,
                "source": doc["filename"],
                "chunk_id": i
            })

print("Total chunks:", len(all_chunks))

Total chunks: 845


In [6]:
# Generate embeddings for document chunks
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

(845, 384)


In [7]:
# Create FAISS vector index
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dimension)

index.add(np.array(embeddings).astype("float32"))

print("Number of vectors:", index.ntotal)

Number of vectors: 845


In [8]:
# Retrieve relevant chunks for a query

def retrieve_chunks(query, k=5):
    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k
    )

    retrieved_chunks = []

    for idx, distance in zip(indices[0], distances[0]):
        chunk = all_chunks[idx]

        retrieved_chunks.append({
            "text": chunk["text"],
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "distance": distance
        })

    return retrieved_chunks

In [9]:
# Build context and prompt for the LLM

query = "How do I prevent problems caused by Varroa?"

retrieved_chunks = retrieve_chunks(query)

context = "\n\n".join(
    chunk["text"] for chunk in retrieved_chunks
)

prompt = f"""
Answer the question using the provided context whenever possible.

If the context contains enough information to answer the question,
answer based on the context and start your response with:
[SUPPORTED]

If the context does not contain enough information to answer the question,
you may answer using your general knowledge. In that case, start your
response with:
[NOT VERIFIED]

When using general knowledge, clearly state that the information is not
verified against the knowledge base.

Do not present information from your general knowledge as if it came from
the provided documents.

Context:
{context}

Question:
{query}
"""

In [10]:
# Generate an answer using Gemini

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

answer = response.text

if answer.startswith("[SUPPORTED]"):
    source_status = "Supported by knowledge base"
    answer = answer.replace("[SUPPORTED]", "", 1).strip()

elif answer.startswith("[NOT VERIFIED]"):
    source_status = "Not verified against knowledge base"
    answer = answer.replace("[NOT VERIFIED]", "", 1).strip()

else:
    source_status = "Source status not determined"

In [11]:
# Display answer and source status

print("Answer:")
print(answer)

print("\nSource status:")
print(source_status)

print("\nRetrieved documents:")

sources = sorted(set(chunk["source"] for chunk in retrieved_chunks))

for source in sources:
    print(f"- {source}")

Answer:
Based on the provided context, you can manage and prevent problems caused by Varroa mites using the following practices:

* **Monitor varroa levels:** Remain vigilant in monitoring varroa levels regularly, both before and after treatments, using information to drive management and treatment decisions.
* **Take timely action:** Be prepared to take timely action to reduce mite loads once one colony reaches the treatment threshold. Treat all colonies in the apiary at the same time.
* **Use a combination of control options:** Use a combination of treatment options and management practices, which include:
  1. Synthetic miticides
  2. Organic miticides
  3. Cultural and mechanical practices
  4. Genetic practices
* **Alternate treatments:** Alternate between different chemical controls and management methods rather than relying on a single product or practice. This helps slow the development of chemical resistance in mites.
* **Follow safety guidelines:** Follow all legal label inst